In [1]:
import sys
sys.path.append('..')
import pandas as pd
import numpy as np
from src.data_loader import DataLoader

# Load raw data
loader = DataLoader(data_dir="../data")
df = pd.read_csv(loader.processed_dir / "raw_data.csv")

In [2]:
def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Comprehensive data cleaning pipeline.
    
    Returns cleaned DataFrame with documentation of changes.
    """
    cleaned_df = df.copy()
    cleaning_log = []
    
    # 1. Handle Missing Values
    for col in cleaned_df.columns:
        missing = cleaned_df[col].isnull().sum()
        if missing > 0:
            if cleaned_df[col].dtype in ['int64', 'float64']:
                median = cleaned_df[col].median()
                cleaned_df[col].fillna(median, inplace=True)
                cleaning_log.append(
                    f"Filled {missing} missing values in '{col}' with median ({median})"
                )
            else:
                mode = cleaned_df[col].mode()[0]
                cleaned_df[col].fillna(mode, inplace=True)
                cleaning_log.append(
                    f"Filled {missing} missing values in '{col}' with mode ('{mode}')"
                )
    
    # 2. Remove Duplicates
    duplicates = cleaned_df.duplicated().sum()
    if duplicates > 0:
        cleaned_df.drop_duplicates(inplace=True)
        cleaning_log.append(f"Removed {duplicates} duplicate rows")
    
    # 3. Fix Data Types
    for col in cleaned_df.columns:
        if cleaned_df[col].dtype == 'object':
            try:
                cleaned_df[col] = pd.to_numeric(cleaned_df[col])
                cleaning_log.append(f"Converted '{col}' to numeric")
            except:
                pass
        
        if cleaned_df[col].nunique() < 10:
            cleaned_df[col] = cleaned_df[col].astype('category')
            cleaning_log.append(f"Converted '{col}' to categorical")
    
    # 4. Handle Outliers (for numeric columns)
    numeric_cols = cleaned_df.select_dtypes(include=['int64', 'float64']).columns
    for col in numeric_cols:
        Q1 = cleaned_df[col].quantile(0.25)
        Q3 = cleaned_df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        outliers = cleaned_df[
            (cleaned_df[col] < lower_bound) |
            (cleaned_df[col] > upper_bound)
        ]
        
        if len(outliers) > 0:
            cleaned_df[col] = cleaned_df[col].clip(lower_bound, upper_bound)
            cleaning_log.append(
                f"Capped {len(outliers)} outliers in '{col}' to [{lower_bound:.2f}, {upper_bound:.2f}]"
            )
    
    # 5. Add Derived Columns
    if 'age' in cleaned_df.columns and 'thalach' in cleaned_df.columns:
        cleaned_df['hr_reserve'] = (220 - cleaned_df['age']) - cleaned_df['thalach']
        cleaning_log.append("Added 'hr_reserve' (estimated max HR [220-age] minus achieved max HR)")
    
    if 'chol' in cleaned_df.columns and 'age' in cleaned_df.columns:
        cleaned_df['chol_age_ratio'] = cleaned_df['chol'] / cleaned_df['age']
        cleaning_log.append("Added 'chol_age_ratio' (cholesterol relative to age)")
    
    # Log cleaning actions
    with open('../reports/cleaning_log.txt', 'w') as f:
        f.write("DATA CLEANING LOG\n")
        f.write("=" * 50 + "\n\n")
        for action in cleaning_log:
            f.write(f"[OK] {action}\n")
        f.write(f"\nFinal shape: {cleaned_df.shape}\n")
    
    return cleaned_df

In [3]:
# Clean the data
df_clean = clean_data(df)

# Save cleaned data
loader.save_processed_data(df_clean, "cleaned_data.csv")

# Show summary of changes
print("\nData cleaning complete!")
print(f"Original shape: {df.shape}")
print(f"Cleaned shape: {df_clean.shape}")
print(f"Check cleaning_log.txt for details")

Saved processed data to ../data/processed/cleaned_data.csv

Data cleaning complete!
Original shape: (1025, 14)
Cleaned shape: (302, 16)
Check cleaning_log.txt for details